In [16]:
from dotenv import load_dotenv

load_dotenv()

True

In [17]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [18]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_ollama import OllamaLLM, ChatOllama

class EmailState(AgentState):
    email: str


model = ChatOllama(
    model="gemma4:31b-cloud",
    temperature=0.7,
    num_predict=256,
)

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [19]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [20]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Please '
                                                                          'let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'time '
                   

In [21]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John, no problem at all. Please let me know what time works best for you to reschedule. Best, Seán.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John, no problem at all. Please let me know what time works best for you to reschedule. Best, Seán.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='c1fe7cf754e8b87e87b6e42d9631835b')]


In [22]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem at all. Please let me know what time works best for you to reschedule. Best, Seán.


## Approve

In [23]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='0c5e41a1-6f3e-46d7-b706-6d37e570be66'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-05-18T07:44:01.098909906Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1494782141, 'load_duration': None, 'prompt_eval_count': 117, 'prompt_eval_duration': None, 'eval_count': 9, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--019e3a0a-af14-7883-bc4a-6a88a2b03ab2-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '4be48405-779e-4c2a-be73-f4a019854daa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 9, 'total_tok

## Reject

In [24]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='0c5e41a1-6f3e-46d7-b706-6d37e570be66'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-05-18T07:44:01.098909906Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1494782141, 'load_duration': None, 'prompt_eval_count': 117, 'prompt_eval_duration': None, 'eval_count': 9, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--019e3a0a-af14-7883-bc4a-6a88a2b03ab2-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '4be48405-779e-4c2a-be73-f4a019854daa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 9, 'total_tok

In [25]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Edit

In [26]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='0c5e41a1-6f3e-46d7-b706-6d37e570be66'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-05-18T07:44:01.098909906Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1494782141, 'load_duration': None, 'prompt_eval_count': 117, 'prompt_eval_duration': None, 'eval_count': 9, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--019e3a0a-af14-7883-bc4a-6a88a2b03ab2-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': '4be48405-779e-4c2a-be73-f4a019854daa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 9, 'total_tok